# TF-QKD Optimization Notebook

This notebook optimizes the TF-QKD key generation rate over a user-defined loss range.

It supports:

- VSS and GS states
- PSO, DE, BO, L-BFGS-B, Nelder-Mead, Powell, or any method exposed in `tfqkd.optimization.optimize`
- optional fixed GS degrees of freedom
- parallel optimization over different `loss_db` points
- automatic saving into `data/optimization_results/run_.../`


In [ ]:
# ============================================================
# 1) User settings
# ============================================================

from __future__ import annotations

import os
import sys
import json
import time
import platform
import traceback
from datetime import datetime
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# Physics / key-rate setup
# -----------------------
f = 1.16
n_truncation = 4
pd = 1e-8

# -----------------------
# Optimization method
# -----------------------
# Supported by the provided optimization.py:
# "PSO", "DE", "BO", "L-BFGS-B", "Nelder-Mead", "Powell"
optimization_method = "PSO"

# -----------------------
# Loss points to optimize
# -----------------------
loss_db_range = np.linspace(0, 30, 7)  # example: 0, 5, 10, ..., 30 dB

# -----------------------
# State type
# -----------------------
# Use "VSS" or "GS"
state_type = "GS"

# -----------------------
# Parallelization over loss points
# -----------------------
# This parallelizes different loss_db points.
# For Windows/Jupyter, start with False. If it works in your setup, set True.
parallel_over_losses = False
max_workers = max(1, min(os.cpu_count() or 1, len(loss_db_range)))

# -----------------------
# Optimizer-specific options
# -----------------------
# Keep small for testing; increase for final runs.
optimizer_kwargs = {
    "n_particles": 24,
    "maxiter": 40,
}

# For DE, useful options would be:
# optimizer_kwargs = {"maxiter": 50, "popsize": 10, "polish": True}

# For BO, install scikit-optimize first:
# pip install scikit-optimize
# optimizer_kwargs = {"n_calls": 50, "n_initial_points": 10}

seed = 1

print("Settings loaded.")
print("state_type:", state_type)
print("method:", optimization_method)
print("loss points:", loss_db_range)


In [ ]:
# ============================================================
# 2) Import local package
# ============================================================

# This notebook should live inside: project_root/notebooks/TF_QKD_optimization.ipynb
# The package should live inside: project_root/src/tfqkd/ or project_root/tfqkd/

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Support both common layouts:
#   project/src/tfqkd
#   project/tfqkd
for p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from tfqkd.optimization import optimize, VSS_BOUNDS, GS_BOUNDS
from tfqkd.key_rate import tf_qkd_fast

print("Project root:", PROJECT_ROOT)
print("Import successful.")


In [ ]:
# ============================================================
# 3) Optional GS degree-of-freedom reduction
# ============================================================

# Full GS vector convention:
# [epsilon, b, phi0, phi1, s, theta0, theta1]

FULL_PARAM_NAMES = {
    "VSS": ["q", "phi0", "phi1"],
    "GS": ["epsilon", "b", "phi0", "phi1", "s", "theta0", "theta1"],
}

FULL_BOUNDS = {
    "VSS": list(VSS_BOUNDS),
    "GS": list(GS_BOUNDS),
}

# For VSS, usually keep all variables free.
# For GS, you can reduce variables by fixing some parameters.
# Example: fix theta0=phi0 and theta1=phi1 is NOT directly supported here because that is a dependency,
# not a constant. For that, use the custom builder section below.

fixed_parameters = {
    # Examples for GS:
    # "epsilon": 0.0,
    # "s": 0.0,
    # "theta0": 0.0,
    # "theta1": np.pi,
}

# Optional: custom dependent constraints for GS.
# Set this True to enforce theta0 = phi0 and theta1 = phi1.
# Then theta0 and theta1 are removed from the optimization variables.
gs_align_squeezing_phase = True

# Set this True to enforce phi1 = phi0 + pi modulo 2pi.
# Then phi1 is removed from the optimization variables.
gs_antipodal_phases = True


def build_reduced_parameterization(state: str, fixed: dict, *, align_squeezing_phase: bool = False, antipodal_phases: bool = False):
    state = state.upper()
    names = FULL_PARAM_NAMES[state]
    bounds = FULL_BOUNDS[state]

    if state != "GS":
        align_squeezing_phase = False
        antipodal_phases = False

    removed = set(fixed.keys())
    if state == "GS" and align_squeezing_phase:
        removed.update(["theta0", "theta1"])
    if state == "GS" and antipodal_phases:
        removed.add("phi1")

    variable_names = [name for name in names if name not in removed]
    variable_bounds = [bounds[names.index(name)] for name in variable_names]

    def expand(x_reduced):
        x_reduced = np.asarray(x_reduced, dtype=float)
        d = {}
        idx = 0
        for name in names:
            if name in fixed:
                d[name] = float(fixed[name])
            elif name in removed:
                # filled below for dependent constraints
                continue
            else:
                d[name] = float(x_reduced[idx])
                idx += 1

        if state == "GS":
            if antipodal_phases:
                d["phi1"] = float((d["phi0"] + np.pi) % (2 * np.pi))
            if align_squeezing_phase:
                d["theta0"] = float(d["phi0"])
                d["theta1"] = float(d["phi1"])

        return np.array([d[name] for name in names], dtype=float)

    return variable_names, variable_bounds, expand


variable_names, reduced_bounds, expand_parameters = build_reduced_parameterization(
    state_type,
    fixed_parameters,
    align_squeezing_phase=gs_align_squeezing_phase,
    antipodal_phases=gs_antipodal_phases,
)

print("Full parameters:", FULL_PARAM_NAMES[state_type.upper()])
print("Optimized variables:", variable_names)
print("Reduced bounds:", reduced_bounds)
print("Fixed parameters:", fixed_parameters)
print("GS align squeezing phase:", gs_align_squeezing_phase)
print("GS antipodal phases:", gs_antipodal_phases)


In [ ]:
# ============================================================
# 4) Reduced objective and optimizer wrapper
# ============================================================

# We reduce variables by optimizing x_reduced, then expanding it into the full parameter vector.
# This lets you fix selected GS parameters while still using the existing tfqkd functions.


def negative_rate_reduced(x_reduced, loss_db, n_truncation, state, pd, f):
    full_x = expand_parameters(x_reduced)
    r, *_ = tf_qkd_fast(
        loss_db=loss_db,
        n_truncation=n_truncation,
        parameters=full_x,
        state=state,
        pd=pd,
        f=f,
    )
    if not np.isfinite(r):
        return 1e20
    return -float(r)


def optimize_one_loss(loss_db: float, run_seed: int | None = None):
    """Optimize one loss point and return a flat result dictionary."""
    from scipy.optimize import differential_evolution, minimize

    method = optimization_method.upper()
    state = state_type.upper()
    run_seed = seed if run_seed is None else run_seed

    t0 = time.time()

    if method == "DE":
        result = differential_evolution(
            negative_rate_reduced,
            bounds=reduced_bounds,
            args=(loss_db, n_truncation, state, pd, f),
            seed=run_seed,
            polish=optimizer_kwargs.get("polish", True),
            maxiter=optimizer_kwargs.get("maxiter", 50),
            popsize=optimizer_kwargs.get("popsize", 10),
            updating="deferred" if optimizer_kwargs.get("workers", 1) != 1 else "immediate",
            workers=optimizer_kwargs.get("workers", 1),
        )
        x_reduced = np.asarray(result.x, dtype=float)
        success = bool(result.success)
        message = str(result.message)

    elif method == "PSO":
        rng = np.random.default_rng(run_seed)

        lower = np.array([b[0] for b in reduced_bounds], dtype=float)
        upper = np.array([b[1] for b in reduced_bounds], dtype=float)
        dim = len(reduced_bounds)

        n_particles = optimizer_kwargs.get("n_particles", 30)
        maxiter = optimizer_kwargs.get("maxiter", 100)
        inertia = optimizer_kwargs.get("inertia", 0.7)
        cognitive = optimizer_kwargs.get("cognitive", 1.5)
        social = optimizer_kwargs.get("social", 1.5)

        positions = rng.uniform(lower, upper, size=(n_particles, dim))
        velocities = 0.1 * (upper - lower) * rng.normal(size=(n_particles, dim))

        def eval_pop(pos):
            return np.array([
                negative_rate_reduced(x, loss_db, n_truncation, state, pd, f)
                for x in pos
            ])

        personal_best = positions.copy()
        personal_best_values = eval_pop(positions)

        best_index = int(np.argmin(personal_best_values))
        global_best = personal_best[best_index].copy()
        global_best_value = float(personal_best_values[best_index])
        history = []

        for _ in range(maxiter):
            r1 = rng.random(size=(n_particles, dim))
            r2 = rng.random(size=(n_particles, dim))

            velocities = (
                inertia * velocities
                + cognitive * r1 * (personal_best - positions)
                + social * r2 * (global_best - positions)
            )
            positions = np.clip(positions + velocities, lower, upper)

            values = eval_pop(positions)
            improved = values < personal_best_values
            personal_best[improved] = positions[improved]
            personal_best_values[improved] = values[improved]

            best_index = int(np.argmin(personal_best_values))
            if float(personal_best_values[best_index]) < global_best_value:
                global_best_value = float(personal_best_values[best_index])
                global_best = personal_best[best_index].copy()

            history.append(-global_best_value)

        x_reduced = global_best
        success = True
        message = "PSO completed"
        result = {"history": history, "fun": global_best_value}

    elif method == "BO":
        try:
            from skopt import gp_minimize
            from skopt.space import Real
        except ImportError as exc:
            raise ImportError("Install Bayesian optimization dependency with: pip install scikit-optimize") from exc

        space = [Real(low, high, name=name) for name, (low, high) in zip(variable_names, reduced_bounds)]

        def objective(x):
            return negative_rate_reduced(np.asarray(x), loss_db, n_truncation, state, pd, f)

        result = gp_minimize(
            objective,
            dimensions=space,
            n_calls=optimizer_kwargs.get("n_calls", 50),
            n_initial_points=optimizer_kwargs.get("n_initial_points", 10),
            random_state=run_seed,
        )
        x_reduced = np.asarray(result.x, dtype=float)
        success = True
        message = "BO completed"

    elif method in {"L-BFGS-B", "NELDER-MEAD", "POWELL"}:
        x0 = optimizer_kwargs.get("x0", None)
        if x0 is None:
            # Use center of reduced bounds if x0 is not provided.
            x0 = np.array([(lo + hi) / 2 for lo, hi in reduced_bounds], dtype=float)
        else:
            x0 = np.asarray(x0, dtype=float)

        use_bounds = method in {"L-BFGS-B", "POWELL"}
        result = minimize(
            negative_rate_reduced,
            x0,
            args=(loss_db, n_truncation, state, pd, f),
            method=method,
            bounds=reduced_bounds if use_bounds else None,
            options={"maxiter": optimizer_kwargs.get("maxiter", 500)},
        )
        x_reduced = np.asarray(result.x, dtype=float)
        success = bool(result.success)
        message = str(result.message)

    else:
        raise ValueError(f"Unknown optimization method: {optimization_method}")

    x_full = expand_parameters(x_reduced)
    rate, e_x, e_z, p_xx = tf_qkd_fast(loss_db, n_truncation, x_full, state, pd=pd, f=f)

    elapsed_s = time.time() - t0

    row = {
        "loss_db": float(loss_db),
        "state": state,
        "method": method,
        "rate": float(rate),
        "e_x": float(e_x),
        "e_z": float(e_z),
        "p_xx": float(p_xx),
        "success": success,
        "message": message,
        "elapsed_s": elapsed_s,
    }

    for name, value in zip(variable_names, x_reduced):
        row[f"var_{name}"] = float(value)

    for name, value in zip(FULL_PARAM_NAMES[state], x_full):
        row[f"full_{name}"] = float(value)

    return row

print("Optimizer wrapper ready.")


In [ ]:
# ============================================================
# 5) Create results folder and save settings
# ============================================================

results_root = PROJECT_ROOT / "data" / "optimization_results"
results_root.mkdir(parents=True, exist_ok=True)

run_name = datetime.now().strftime(f"run_%Y%m%d_%H%M%S_{state_type.upper()}_{optimization_method.upper()}")
run_dir = results_root / run_name
run_dir.mkdir(parents=True, exist_ok=False)

settings = {
    "run_name": run_name,
    "timestamp": datetime.now().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "state_type": state_type.upper(),
    "f": f,
    "n_truncation": n_truncation,
    "pd": pd,
    "optimization_method": optimization_method,
    "loss_db_range": list(map(float, loss_db_range)),
    "parallel_over_losses": parallel_over_losses,
    "max_workers": max_workers,
    "optimizer_kwargs": optimizer_kwargs,
    "seed": seed,
    "fixed_parameters": fixed_parameters,
    "gs_align_squeezing_phase": gs_align_squeezing_phase,
    "gs_antipodal_phases": gs_antipodal_phases,
    "variable_names": variable_names,
    "reduced_bounds": reduced_bounds,
    "full_parameter_names": FULL_PARAM_NAMES[state_type.upper()],
    "python_version": sys.version,
    "platform": platform.platform(),
}

# Save as JSON and CSV. The user asked for CSV; JSON is also useful for nested settings.
with open(run_dir / "settings.json", "w", encoding="utf-8") as fjson:
    json.dump(settings, fjson, indent=2)

settings_rows = []
for key, value in settings.items():
    settings_rows.append({"setting": key, "value": json.dumps(value) if isinstance(value, (list, dict, tuple)) else value})
settings_df = pd.DataFrame(settings_rows)
settings_df.to_csv(run_dir / "settings.csv", index=False)

print("Run directory created:", run_dir)
print("Saved settings.csv and settings.json")


In [ ]:
# ============================================================
# 6) Run optimization loop over loss_db points
# ============================================================

all_rows = []
failed_rows = []

t0_total = time.time()

if parallel_over_losses:
    print(f"Running in parallel over losses with max_workers={max_workers}")
    print("Note: On Windows/Jupyter, multiprocessing may fail depending on environment. If it fails, set parallel_over_losses=False.")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_loss = {
            executor.submit(optimize_one_loss, float(loss_db), None if seed is None else seed + i): float(loss_db)
            for i, loss_db in enumerate(loss_db_range)
        }

        for future in as_completed(future_to_loss):
            loss_db = future_to_loss[future]
            try:
                row = future.result()
                all_rows.append(row)
                print(f"Done loss={loss_db:.3f} dB | rate={row['rate']:.6e} | elapsed={row['elapsed_s']:.2f}s")
            except Exception as exc:
                tb = traceback.format_exc()
                failed_rows.append({"loss_db": loss_db, "error": str(exc), "traceback": tb})
                print(f"FAILED loss={loss_db:.3f} dB: {exc}")

else:
    print("Running serial loop over losses.")
    for i, loss_db in enumerate(loss_db_range):
        loss_db = float(loss_db)
        try:
            row = optimize_one_loss(loss_db, None if seed is None else seed + i)
            all_rows.append(row)
            print(f"Done loss={loss_db:.3f} dB | rate={row['rate']:.6e} | elapsed={row['elapsed_s']:.2f}s")
        except Exception as exc:
            tb = traceback.format_exc()
            failed_rows.append({"loss_db": loss_db, "error": str(exc), "traceback": tb})
            print(f"FAILED loss={loss_db:.3f} dB: {exc}")

elapsed_total = time.time() - t0_total
print(f"Total elapsed time: {elapsed_total:.2f} s")


In [ ]:
# ============================================================
# 7) Save optimization results
# ============================================================

results_df = pd.DataFrame(all_rows).sort_values("loss_db") if all_rows else pd.DataFrame()
failed_df = pd.DataFrame(failed_rows)

results_csv = run_dir / "optimized_results.csv"
failed_csv = run_dir / "failed_results.csv"

results_df.to_csv(results_csv, index=False)
if not failed_df.empty:
    failed_df.to_csv(failed_csv, index=False)

print("Saved:", results_csv)
if not failed_df.empty:
    print("Saved failures:", failed_csv)

results_df


In [ ]:
# ============================================================
# 8) Plot and save key-rate curve
# ============================================================

if not results_df.empty:
    plt.figure(figsize=(7, 5))
    plt.semilogy(results_df["loss_db"], results_df["rate"], marker="o")
    plt.xlabel("Loss (dB)")
    plt.ylabel("Optimized key generation rate")
    plt.title(f"TF-QKD optimization: {state_type.upper()} / {optimization_method.upper()}")
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()

    fig_path = run_dir / "key_rate_vs_loss.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()

    print("Saved plot:", fig_path)
else:
    print("No successful results to plot.")


In [ ]:
# ============================================================
# 9) Quick reload check
# ============================================================

loaded_settings = pd.read_csv(run_dir / "settings.csv")
loaded_results = pd.read_csv(run_dir / "optimized_results.csv") if (run_dir / "optimized_results.csv").exists() else pd.DataFrame()

print("Run folder:", run_dir)
print("Settings rows:", len(loaded_settings))
print("Result rows:", len(loaded_results))

loaded_results.head()
